# Mat & Comp – ML 6: Agrupamiento con k-means
## Encontrar grupos sin que nadie diga las respuestas

**Curso:** Machine learning (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 5

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Distinguir el aprendizaje **supervisado** (con respuestas) del **no supervisado** (sin respuestas).
2. Programar desde cero el algoritmo **k-means**: asignar cada punto al centro más cercano y mover los centros al promedio.
3. Medir la calidad de un agrupamiento con la **inercia** y elegir k con el **método del codo**.
4. Reconocer que k-means depende del punto de partida, y cómo se soluciona.
5. Usar `KMeans` de **scikit-learn** para agrupar estudiantes, dígitos escritos a mano y colores de una foto.

## 🔍 ¿Dónde ponemos las bodegas?

Una empresa de domicilios de Ibagué tiene 75 clientes frecuentes y quiere abrir **3 bodegas**, de modo que cada cliente quede lo más cerca posible de alguna. Nadie le dice qué clientes van con qué bodega: eso es justamente lo que hay que descubrir.

✏️ *Mira el mapa de la sección 1. ¿Dónde pondrías las 3 bodegas? Marca tres puntos con el dedo en la pantalla.*

En ML 4 y ML 5 siempre teníamos la **respuesta correcta** de cada dato (mandarina o limón, aprobó o no) y el modelo aprendía de ella: eso se llama aprendizaje **supervisado**. Hoy **no hay respuestas**. Solo hay puntos, y queremos que la computadora descubra sola qué grupos forman. Eso se llama aprendizaje **no supervisado**, y el problema se llama **agrupamiento** (*clustering*).

## 1. Los datos: clientes en un mapa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
zonas = np.array([[2, 3], [7, 8], [8, 2.5]])            # solo para generar los datos
P = np.vstack([rng.normal(z, 0.9, (25, 2)) for z in zonas]).round(2)

print("Forma:", P.shape, "→ 75 clientes, cada uno con coordenadas (x, y) en km")

plt.figure(figsize=(6, 6))
plt.scatter(P[:, 0], P[:, 1], color="gray", s=30)
plt.xlabel("x (km)")
plt.ylabel("y (km)")
plt.title("75 clientes, sin etiquetas")
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()

Para nosotros es fácil ver tres grupos. Pero la computadora solo tiene una tabla de 75 × 2 números. ¿Cómo los descubre?

## 2. El algoritmo k-means, paso a paso

**k-means** (k medias) busca **k centros** de modo que cada punto quede cerca de su centro. Funciona repitiendo dos pasos muy sencillos:

0. **Empezar:** elegir k puntos al azar como centros iniciales.
1. **Asignar:** cada punto se une al grupo del centro **más cercano** (distancia de Pitágoras, como en ML 4).
2. **Mover:** cada centro se muda al **promedio** (la «media») de los puntos de su grupo. De ahí el nombre.
3. **Repetir** los pasos 1 y 2 hasta que los centros ya no se muevan.

Hagámoslo a mano con k = 3. Primero el paso 0:

In [ ]:
k = 3
r = np.random.default_rng(0)
centros = P[r.choice(len(P), k, replace=False)]      # 3 clientes al azar, sin repetir
print("Centros iniciales:")
print(centros)

**Paso 1: asignar.** Calculamos la distancia de los 75 puntos a cada centro. Cada centro da una columna de 75 distancias; `np.column_stack` las junta en una tabla de 75 × 3. Luego `argmin(axis=1)` nos dice, en cada fila, cuál columna (cuál centro) tiene la distancia más pequeña.

In [ ]:
def asignar(P, centros):
    distancias = np.column_stack([np.sqrt(((P - c) ** 2).sum(axis=1)) for c in centros])
    return distancias.argmin(axis=1)          # para cada punto: 0, 1 o 2

grupos = asignar(P, centros)
print("Grupo de los primeros 10 clientes:", grupos[:10])
print("Tamaño de cada grupo:", np.bincount(grupos))

💡 `np.bincount` cuenta cuántas veces aparece cada número 0, 1, 2…

**Paso 2: mover.** Cada centro se va al promedio de su grupo. Con una máscara (`grupos == j`) tomamos los puntos del grupo j y con `mean(axis=0)` promediamos sus x y sus y.

In [ ]:
def mover(P, grupos, k):
    return np.array([P[grupos == j].mean(axis=0) for j in range(k)])

nuevos = mover(P, grupos, k)
print("Antes:")
print(centros.round(2))
print("Después:")
print(nuevos.round(2))

Ahora repetimos y dibujamos cómo se van moviendo los centros. Las cruces negras son los centros, y los colores, los grupos:

In [ ]:
colores = np.array(["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"])

centros = P[np.random.default_rng(0).choice(len(P), 3, replace=False)]
pasos_a_dibujar = [0, 1, 5, 6, 7, 8]
fig, ejes = plt.subplots(2, 3, figsize=(15, 10))
ejes = ejes.ravel()
for paso in range(9):
    grupos = asignar(P, centros)
    if paso in pasos_a_dibujar:
        eje = ejes[pasos_a_dibujar.index(paso)]
        eje.scatter(P[:, 0], P[:, 1], c=colores[grupos], s=25)
        eje.scatter(centros[:, 0], centros[:, 1], color="black", marker="X", s=200)
        eje.set_title(f"Paso {paso}")
        eje.set_aspect("equal")
    centros = mover(P, grupos, 3)
plt.show()

Dibujamos solo algunos pasos (del 2 al 4 casi nada cambia). ✏️ *Observa el paso 0: dos centros empiezan en el mismo grupo de arriba y uno solo carga con los dos grupos de abajo. ¿Cómo logra el algoritmo «escapar» de ese mal comienzo? ¿En qué paso dejan de cambiar los colores?*

## 3. ¿Qué tan bueno es un agrupamiento? La inercia

Para medir qué tan compactos son los grupos usamos la **inercia**: la suma de las distancias **al cuadrado** de cada punto a su centro. Es la misma idea del error cuadrático de ML 2: cuanto más pequeña, mejor.

$$\text{inercia} = \sum_{\text{puntos}} (\text{distancia al centro de su grupo})^2$$

Juntemos todo en una función que además guarde la inercia en cada paso:

In [ ]:
def inercia(P, grupos, centros):
    return ((P - centros[grupos]) ** 2).sum()

def kmeans(P, k, semilla, max_pasos=50):
    centros = P[np.random.default_rng(semilla).choice(len(P), k, replace=False)]
    historia = []
    for paso in range(max_pasos):
        grupos = asignar(P, centros)
        historia.append(inercia(P, grupos, centros))
        nuevos = mover(P, grupos, k)
        if np.allclose(nuevos, centros):       # los centros ya no se mueven: terminamos
            break
        centros = nuevos
    return grupos, centros, historia

grupos, centros, historia = kmeans(P, 3, semilla=0)
print("Inercia en cada paso:", np.round(historia, 1))

plt.plot(historia, "o-")
plt.xlabel("paso")
plt.ylabel("inercia")
plt.title("La inercia baja en cada paso")
plt.grid(alpha=0.3)
plt.show()

💡 `centros[grupos]` arma una tabla de 75 filas donde cada fila es el centro del grupo de ese punto. Así restamos todo de una vez.

La inercia **nunca sube**: asignar al centro más cercano y mover al promedio solo pueden acercar los puntos a sus centros. Por eso el algoritmo siempre termina.

## 4. ⚠️ Un mal comienzo

¿Siempre llega a la misma respuesta? Probemos varias semillas (varios comienzos al azar):

In [ ]:
for semilla in range(6):
    g, c, h = kmeans(P, 3, semilla)
    print(f"semilla {semilla}: inercia final {h[-1]:7.1f}   tamaños {np.bincount(g, minlength=3)}   pasos {len(h)}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
for eje, semilla in zip(ejes, [0, 3]):
    g, c, h = kmeans(P, 3, semilla)
    eje.scatter(P[:, 0], P[:, 1], c=colores[g], s=25)
    eje.scatter(c[:, 0], c[:, 1], color="black", marker="X", s=200)
    eje.set_title(f"semilla {semilla}: inercia {h[-1]:.1f}")
    eje.set_aspect("equal")
plt.show()

Con la semilla 3, dos centros empezaron en el mismo grupo y se quedaron «atrapados» repartiéndoselo, mientras que un solo centro carga con dos grupos. Es como el descenso de gradiente de ML 3 cayendo en un valle que no es el más profundo: k-means encuentra un **mínimo local**.

**La solución:** correr k-means varias veces con distintos comienzos y quedarse con el resultado de **menor inercia**. scikit-learn lo hace automáticamente con el parámetro `n_init`.

## 5. ¿Cuántos grupos? El método del codo

En la vida real nadie nos dice cuánto vale k. ¿Por qué no probar con muchos grupos? Porque la inercia **siempre** baja al aumentar k: con 75 grupos (un centro en cada cliente) la inercia sería 0, ¡pero no habríamos descubierto nada!

Grafiquemos la mejor inercia (de 10 comienzos) para cada k:

In [ ]:
ks = range(1, 9)
mejores = []
for k in ks:
    mejores.append(min(kmeans(P, k, s)[2][-1] for s in range(10)))
    print(f"k = {k}: inercia {mejores[-1]:.1f}")

plt.plot(list(ks), mejores, "o-")
plt.xlabel("k (número de grupos)")
plt.ylabel("inercia")
plt.title("El método del codo")
plt.grid(alpha=0.3)
plt.show()

La inercia baja muchísimo hasta k = 3 y después baja poco. Ese quiebre se llama **codo**, y sugiere que 3 es un buen número de grupos: agregar más bodegas ya casi no acerca a los clientes.

✏️ *Si cada bodega costara mucho dinero, ¿cuántas abrirías? ¿Y si fueran baratas?*

## 6. Con scikit-learn

`KMeans` funciona como los modelos de ML 4 y ML 5, con una diferencia: `fit` recibe solo `X`, **sin `y`**, porque no hay respuestas.

In [ ]:
from sklearn.cluster import KMeans

modelo = KMeans(n_clusters=3, n_init=10, random_state=0)
modelo.fit(P)

print("Centros (bodegas):")
print(modelo.cluster_centers_.round(2))
print("Inercia:", round(modelo.inertia_, 1))
print("Grupo de los primeros 10 clientes:", modelo.labels_[:10])

Llega la misma inercia que nuestra versión. Los números de los grupos pueden estar en otro orden: k-means no sabe qué grupo es «el primero», solo cuáles puntos van juntos.

Si llega un cliente nuevo, `predict` dice a qué bodega le corresponde (la más cercana):

In [ ]:
nuevos_clientes = [[5, 5], [1, 1], [9, 9]]
print(modelo.predict(nuevos_clientes))

## 7. Datos reales: ¿qué tipos de estudiantes hay?

Agrupemos a los estudiantes de ML 1 según sus horas de **estudio** y de **redes sociales**, sin mirar sus notas. Como en ML 4, **normalizamos** primero, porque k-means usa distancias.

In [ ]:
import pandas as pd

datos = pd.read_csv("https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv").dropna()
X = datos[["horas_estudio", "horas_redes"]].to_numpy()
Xn = (X - X.mean(axis=0)) / X.std(axis=0)

modelo_e = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xn)
datos["grupo"] = modelo_e.labels_

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=colores[datos["grupo"]], s=50)
plt.xlabel("horas de estudio por semana")
plt.ylabel("horas de redes por día")
plt.title("3 grupos de estudiantes encontrados por k-means")
plt.show()

k-means no sabe qué significan los grupos: solo dice cuáles estudiantes se parecen. Para **interpretarlos**, miremos el promedio de cada grupo, incluyendo variables que **no** usamos para agrupar, como el sueño y la nota (`groupby` de ML 1):

In [ ]:
resumen = datos.groupby("grupo")[["horas_estudio", "horas_redes", "horas_sueno", "nota_matematicas"]].mean().round(2)
resumen["estudiantes"] = datos["grupo"].value_counts()
resumen

✏️ *Ponle un nombre a cada grupo. ¿Qué pasa con el sueño y la nota del grupo que más usa redes, aunque k-means nunca miró esas columnas?*

⚠️ **Cuidado al interpretar:** los grupos dependen de las variables que elegimos, de k y de la normalización. No son «tipos de personas» que existan en la naturaleza: son una forma útil de resumir los datos.

## 8. Dígitos sin etiquetas

scikit-learn trae 1797 imágenes de dígitos escritos a mano de 8 × 8 píxeles (las usamos en el reto de ML 4). ¿Qué pasa si le pedimos a k-means 10 grupos **sin decirle** qué número es cada imagen? Cada centro es el promedio de muchas imágenes, así que también es una imagen: ¡podemos dibujarlo!

In [ ]:
from sklearn.datasets import load_digits

digitos = load_digits()
modelo_d = KMeans(n_clusters=10, n_init=10, random_state=0).fit(digitos.data)

fig, ejes = plt.subplots(1, 10, figsize=(15, 2))
for j, eje in enumerate(ejes):
    eje.imshow(modelo_d.cluster_centers_[j].reshape(8, 8), cmap="gray_r")
    eje.set_title(f"grupo {j}")
    eje.axis("off")
plt.show()

¡Los centros parecen dígitos! Nadie le dijo a k-means qué es un «3» o un «7»: descubrió que las imágenes de un mismo número se parecen entre sí. Veamos qué tan «puro» es cada grupo, comparando con las etiquetas verdaderas (que k-means nunca vio):

In [ ]:
for j in range(10):
    reales = digitos.target[modelo_d.labels_ == j]
    mas_comun = np.bincount(reales).argmax()
    pureza = np.mean(reales == mas_comun)
    print(f"grupo {j}: {len(reales):3} imágenes, la mayoría son el {mas_comun} ({pureza:.0%})")

Algunos números quedan muy bien separados (0, 4 y 6 casi perfectos) y otros se mezclan: hay un grupo con unos cien 1 y cien 8 juntos, y ningún grupo tiene mayoría de 8. Es impresionante para un algoritmo que no tenía ninguna respuesta.

## 9. Errores comunes

**Error 1: olvidar normalizar.** Igual que en k-NN: si una variable tiene números mucho más grandes, domina las distancias y los grupos se forman casi solo con ella.

**Error 2: pasar columnas de texto.** k-means solo entiende números:

In [ ]:
KMeans(n_clusters=3, n_init=10).fit(datos[["horas_estudio", "deporte"]])

**Error 3: creer que los números de grupo significan algo.** El «grupo 0» no es mejor ni peor que el «grupo 2»; si cambias la semilla, los números pueden cambiar de orden.

**Error 4: esperar que k-means encuentre cualquier forma.** k-means asigna cada punto al centro más cercano, así que siempre forma grupos «redondos» de tamaño parecido. Si los grupos reales tienen forma de anillo o de media luna, se equivoca (lo verás en «¿Qué pasa si…?»).

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …corres `kmeans(P, 1, 0)`? ¿Dónde queda el único centro?
2. …corres `KMeans(n_clusters=75, n_init=1).fit(P)`? ¿Cuánto vale la inercia?
3. …cambias `random_state` en el `KMeans` de los estudiantes? ¿Cambian los grupos o solo sus números?
4. …agrupas dos medias lunas? Prueba con:
   ```python
   from sklearn.datasets import make_moons
   Xm, _ = make_moons(200, noise=0.05, random_state=0)
   gm = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xm)
   plt.scatter(Xm[:, 0], Xm[:, 1], c=gm)
   ```
5. …usas `KMeans(n_clusters=3, n_init=1, random_state=…)` con los clientes y pruebas varios valores de `random_state`? ¿Aparece alguna vez el mal resultado de la sección 4?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). A mano en una recta.** Los puntos son 1, 2, 3, 10, 11 y 12, y empezamos con k = 2 y centros en 1 y 2. Con lápiz: asigna cada punto al centro más cercano, mueve los centros al promedio y repite hasta que no cambien. ¿Dónde terminan los centros? Compruébalo con `kmeans` (pista: `np.array([[1], [2], [3], [10], [11], [12]])`; para empezar exactamente en 1 y 2 tendrás que escribir tú los centros iniciales).

**Ejercicio 2 (fácil). Cuatro bodegas.** Agrupa a los clientes con k = 4 usando `KMeans` y dibuja el resultado con los centros. ¿Qué grupo se partió en dos? ¿Cuánto bajó la inercia comparado con k = 3?

**Ejercicio 3 (medio). Tres variables.** Agrupa a los estudiantes con horas de estudio, sueño y redes (normalizadas) y k = 3. Haz el resumen con `groupby`. ¿Se parecen los grupos a los de la sección 7?

**Ejercicio 4 (medio). El codo de los estudiantes.** Dibuja la gráfica del codo (inercia contra k, de 1 a 8) para los estudiantes normalizados de la sección 7, usando `KMeans(...).inertia_`. ¿Se ve un codo tan claro como con los clientes? ¿Qué k elegirías?

**Ejercicio 5 (medio). Grupos y respuestas.** Usa los grupos de la sección 7 para «predecir» si un estudiante aprobó: en cada grupo, predice lo que sea más común en él (aprobó o no). ¿Qué precisión logras? Compárala con el árbol de profundidad 2 de ML 5 (93 %). ¿Por qué tiene sentido que sea menor?

## 🏆 Reto opcional: una foto con pocos colores

Una foto a color es una tabla de píxeles, y cada píxel tiene 3 números (rojo, verde y azul, de 0 a 255). ¡Cada píxel es un punto en 3 dimensiones! Si agrupamos los píxeles con k-means y pintamos cada uno con el color de su centro, la foto queda con solo k colores.

```python
from sklearn.datasets import load_sample_image
foto = load_sample_image("flower.jpg")           # 427 × 640 píxeles × 3 colores
pixeles = foto.reshape(-1, 3) / 255               # una fila por píxel
```

1. Toma una muestra de 5000 píxeles al azar y entrena `KMeans` con k = 4, 8 y 16 sobre la muestra.
2. Con `predict`, asigna **todos** los píxeles a su centro y reconstruye la foto con `centros[grupos].reshape(foto.shape)`.
3. Muestra las fotos lado a lado con `plt.imshow`. ¿Con cuántos colores ya casi no notas la diferencia? Así funcionaban los formatos de imagen con paleta, como el GIF.

## 🎬 Cierre

Hoy aprendiste que:
- En el aprendizaje **no supervisado** no hay respuestas: la computadora descubre la estructura de los datos.
- **k-means** repite dos pasos: asignar cada punto al centro más cercano y mover cada centro al promedio de su grupo.
- La **inercia** mide qué tan compactos son los grupos; nunca sube durante el algoritmo, pero puede quedarse en un mínimo local. Por eso se corre varias veces (`n_init`).
- El **método del codo** ayuda a elegir k.
- Los grupos se **interpretan** mirando sus promedios, con cuidado.

**Para pensar:** en ML 4 y ML 5 medimos qué tan buenos eran los modelos «dejando uno afuera», y vimos que medir con los mismos datos de entrenamiento engaña. En la próxima lección ordenaremos todas estas ideas: **datos de entrenamiento y de prueba**, la **matriz de confusión** y cómo detectar y evitar el **sobreajuste**. 🎯